# Arrays irregulares, desiguales, Awkward Arrays

![awkward](img/awkward_logo.png)

# ¿Qué es Awkward Array?

La lección anterior incluía una rebanada complicada:

```python
corte = muones["nMuon"] == 2

pt0 = muones["Muon_pt", corte, 0]
```

Las tres partes de la rebanada `muones["Muon_pt", corte, 0]`

1. seleccionan el campo `"Muon_pt"` de todos los registros del array,
2. aplican `corte`, un array booleano, para seleccionar solo los eventos con dos muones,
3. seleccionan el primer (`0`) muón de cada uno de esos pares. De manera similar para los segundos (`1`) muones.

NumPy no sería capaz de realizar una rebanada así, ni siquiera de representar un array de listas de longitud variable sin recurrir a arrays de objetos.

In [ ]:
import numpy as np

# genera un ValueError
np.array([[0.0, 1.1, 2.2], [], [3.3, 4.4], [5.5], [6.6, 7.7, 8.8, 9.9]])

Awkward Array está diseñado para llenar este vacío:

In [ ]:
import awkward as ak

ak.Array([[0.0, 1.1, 2.2], [], [3.3, 4.4], [5.5], [6.6, 7.7, 8.8, 9.9]])

A los arrays como este se les llama a veces "arrays irregulares" o "desiguales" (en inglés, "[jagged arrays](https://en.wikipedia.org/wiki/Jagged_array)" o "ragged arrays").

## Rebanadas en Awkward Array

Las rebanadas básicas son una generalización de las de NumPy: lo que NumPy haría si tuviera listas de longitud variable.

In [ ]:
array = ak.Array([[0.0, 1.1, 2.2], [], [3.3, 4.4], [5.5], [6.6, 7.7, 8.8, 9.9]])
array.tolist()

In [ ]:
array[2]

In [ ]:
array[-1, 1]

In [ ]:
array[2:, 0]

In [ ]:
array[2:, 1:]

In [ ]:
array[:, 0]

**Quiz rápido:** ¿por qué la última genera un error?

Las rebanadas con booleanos y con enteros también funcionan:

In [ ]:
array[[True, False, True, False, True]]

In [ ]:
array[[2, 3, 3, 1]]

Igual que en NumPy, se pueden calcular arrays booleanos para las rebanadas, y funciones como [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html) son útiles para eso.

In [ ]:
ak.num(array)

In [ ]:
ak.num(array) > 0

In [ ]:
array[ak.num(array) > 0, 0]

In [ ]:
array[ak.num(array) > 1, 1]

Ahora considera esto (similar a un ejemplo de la primera lección):

In [ ]:
corte = array * 10 % 2 == 0

array[corte]

Este array, `corte`, no es solo un array de booleanos. Es un array irregular de booleanos. Todas sus listas anidadas encajan en las listas anidadas de `array`, por lo que puede seleccionar números en profundidad, en lugar de seleccionar listas.

## Aplicación: seleccionar partículas, en lugar de eventos

Volviendo al TTree grande de la lección anterior,

In [ ]:
import uproot

url_archivo = "root://eospublic.cern.ch//eos/opendata/cms/derived-data/AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root"

# Si estás en Windows o no tienes XRootD instalado, puedes usar esta url en su lugar
# url_archivo = "https://root.cern/files/rootbench/Run2012BC_DoubleMuParked_Muons.root"

archivo = uproot.open(url_archivo)
tree = archivo["Events"]

muon_pt = tree["Muon_pt"].array(entry_stop=10)

Este array irregular de booleanos selecciona todos los *muones* con más de 20 GeV:

In [ ]:
corte_particula = muon_pt > 20

muon_pt[corte_particula]

y este array de booleanos no irregular (hecho con [ak.any](https://awkward-array.org/doc/main/reference/generated/ak.any.html)) selecciona todos los eventos *que tienen* un muón con más de 20 GeV:

In [ ]:
corte_evento = ak.any(muon_pt > 20, axis=1)

muon_pt[corte_evento]

**Quiz rápido:** construye exactamente el mismo `corte_evento` usando [ak.max](https://awkward-array.org/doc/main/reference/generated/ak.max.html).

**Quiz rápido:** aplica ambos cortes; es decir, selecciona los muones con más de 20 GeV de los eventos que los tienen.

Sugerencia: vas a querer construir un

```python
seleccionados = muon_pt[corte_particula]
```

intermedio, y no puedes usar la variable `corte_evento` tal como está.


Sugerencia: el resultado final debería ser un array irregular, igual que `muon_pt`, pero con menos listas y menos elementos en esas listas.

````{note}
:class: dropdown
## Solución (¡no hagas trampa!)

```python
seleccionados = muon_pt[corte_particula]
resultado_final = seleccionados[corte_evento]
```
````

# Combinatoria en Awkward Array

Las listas de longitud variable presentan más problemas que solo el rebanado y el cálculo de fórmulas array por array. A menudo queremos combinar partículas en todos los pares posibles (dentro de cada evento) para buscar cadenas de desintegración.

## Pares a partir de dos arrays, pares a partir de un solo array

Awkward Array tiene funciones que generan estas combinaciones. Por ejemplo, [ak.cartesian](https://awkward-array.org/doc/main/reference/generated/ak.cartesian.html) toma un producto cartesiano por evento (cuando `axis=1`, el valor por omisión).

![cartoon-cartesian](img/cartoon-cartesian.png)

In [ ]:
numeros = ak.Array([[1, 2, 3], [], [5, 7], [11]])
letras = ak.Array([["a", "b"], ["c"], ["d"], ["e", "f"]])

pares = ak.cartesian((numeros, letras))

Estos `pares` son 2-tuplas, que se parecen a registros en la forma en que se extraen de un array: usando cadenas.

In [ ]:
pares["0"]

In [ ]:
pares["1"]

También existe [ak.unzip](https://awkward-array.org/doc/main/reference/generated/ak.unzip.html), que extrae cada campo en un array separado (lo opuesto de [ak.zip](https://awkward-array.org/doc/main/reference/generated/ak.zip.html)).

In [ ]:
izquierda, derecha = ak.unzip(pares)
izquierda

In [ ]:
derecha

Ten en cuenta que estos `izquierda` y `derecha` no son los `numeros` y `letras` originales: han sido duplicados y tienen la misma forma.

El producto cartesiano es equivalente a este bucle `for` de C++ sobre dos colecciones:

```cpp
for (int i = 0; i < numeros.size(); i++) {
  for (int j = 0; j < letras.size(); j++) {
    // calcular la fórmula con numeros[i] y letras[j]
  }
}
```

A veces, sin embargo, queremos encontrar todos los pares dentro de una sola colección, sin repetición. Eso sería equivalente a este bucle `for` de C++:

```cpp
for (int i = 0; i < numeros.size(); i++) {
  for (int j = i + 1; j < numeros.size(); j++) {
    // calcular la fórmula con numeros[i] y numeros[j]
  }
}
```

La función de Awkward para este caso es [ak.combinations](https://awkward-array.org/doc/main/reference/generated/ak.combinations.html).

![cartoon-combinations](img/cartoon-combinations.png)

In [ ]:
pares = ak.combinations(numeros, 2)
pares

izquierda, derecha = ak.unzip(pares)

izquierda * derecha  # se alinean, así que podemos calcular fórmulas

## Aplicación a los dimuones

La búsqueda de dimuones de la lección anterior fue un poco ingenua, en el sentido de que exigíamos que existieran *exactamente dos* muones en cada evento y solo calculábamos la masa de esa combinación. Si hubiera un tercer muón presente porque se trata de una desintegración electrodébil compleja o porque algo se midió mal, seríamos ciegos a los otros dos muones. Podrían ser dimuones reales.

Un mejor procedimiento sería buscar todos los pares de muones en un evento y aplicar algún criterio para seleccionarlos.

En este ejemplo, juntaremos con [ak.zip](https://awkward-array.org/doc/main/reference/generated/ak.zip.html) las variables de los muones en registros.

In [ ]:
import uproot
import awkward as ak

url_archivo = "root://eospublic.cern.ch//eos/opendata/cms/derived-data/AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root"

# Si estás en Windows o no tienes XRootD instalado, puedes usar esta url en su lugar
# url_archivo = "https://root.cern/files/rootbench/Run2012BC_DoubleMuParked_Muons.root"

archivo = uproot.open(url_archivo)
tree = archivo["Events"]

arrays = tree.arrays(filter_name="/Muon_(pt|eta|phi|charge)/", entry_stop=10000)

muones = ak.zip(
    {
        "pt": arrays["Muon_pt"],
        "eta": arrays["Muon_eta"],
        "phi": arrays["Muon_phi"],
        "charge": arrays["Muon_charge"],
    }
)

In [ ]:
arrays.type

In [ ]:
muones.type

La diferencia entre `arrays` y `muones` es que `arrays` contiene listas separadas de `"Muon_pt"`, `"Muon_eta"`, `"Muon_phi"`, `"Muon_charge"`, mientras que `muones` contiene listas de registros con los campos `"pt"`, `"eta"`, `"phi"`, `"charge"`.

Ahora podemos calcular pares de *objetos* muón

In [ ]:
pares = ak.combinations(muones, 2)

pares.type

y separarlos en arrays del primer muón y del segundo muón de cada par.

In [ ]:
mu1, mu2 = ak.unzip(pares)

**Quiz rápido:** ¿cómo garantizarías que todas las listas de registros en `mu1` y `mu2` tengan las mismas longitudes? Sugerencia: consulta [ak.num](https://awkward-array.org/doc/main/reference/generated/ak.num.html) y [ak.all](https://awkward-array.org/doc/main/reference/generated/ak.all.html).

Dado que sí tienen las mismas longitudes, podemos usarlos en una fórmula.

In [ ]:
import numpy as np

masa = np.sqrt(
    2 * mu1.pt * mu2.pt * (np.cosh(mu1.eta - mu2.eta) - np.cos(mu1.phi - mu2.phi))
)

**Quiz rápido:** ¿cuántas masas tenemos en cada evento? ¿Cómo se compara esto con `muones`, `mu1` y `mu2`?

## Graficar el array irregular

Dado que esta `masa` es un array irregular, no se puede histogramar directamente. Los histogramas toman un conjunto de *números* como entrada, pero este array contiene *listas*.

Suponiendo que solo quieres graficar los números de las listas, puedes usar [ak.flatten](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html) para aplanar un nivel de listas, o [ak.ravel](https://awkward-array.org/doc/main/reference/generated/ak.ravel.html) para aplanar todos los niveles.

In [ ]:
import hist

hist.Hist(hist.axis.Regular(120, 0, 120, label="masa [GeV]")).fill(
    ak.ravel(masa)
).plot();

Alternativamente, supongamos que quieres graficar la masa candidata *máxima* de cada evento, sesgándola hacia los bosones Z. [ak.max](https://awkward-array.org/doc/main/reference/generated/ak.max.html) es una función diferente que selecciona un elemento de cada lista, cuando se usa con `axis=1`.

In [ ]:
ak.max(masa, axis=1)

Algunos valores son `None` porque no hay máximo de una lista vacía. Llamar a [ak.flatten](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html) con `axis=0` elimina estos valores faltantes,

In [ ]:
ak.flatten(ak.max(masa, axis=1), axis=0)

pero también lo hace eliminar las listas vacías desde el principio.

In [ ]:
ak.max(masa[ak.num(masa) > 0], axis=1)

Ten en cuenta que aquí [ak.ravel](https://awkward-array.org/doc/main/reference/generated/ak.ravel.html) *no* es intercambiable con [ak.flatten](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html): aplana todos los niveles de anidamiento, pero conserva los valores faltantes, así que `ak.ravel(ak.max(masa, axis=1))` todavía contiene `None`. Para eliminarlos, usa [ak.drop_none](https://awkward-array.org/doc/main/reference/generated/ak.drop_none.html). Vas a necesitar esto en el Ejercicio 3.

`````{tip}
# Ejercicio 2 (5 minutos)

Selecciona pares de muones con cargas opuestas. Este no es un corte a nivel de evento ni un corte a nivel de partícula, es un corte sobre *pares* de partículas.

````{note}
:class: dropdown
# Solución (¡no hagas trampa!)
Las variables `mu1` y `mu2` son las mitades izquierda y derecha de los pares de muones. Por lo tanto,

```python
corte = (mu1.charge != mu2.charge)
```
tiene la multiplicidad correcta para aplicarse al array `masa`.

```python
hist.Hist(hist.axis.Regular(120, 0, 120, label="masa [GeV]")).fill(
    ak.ravel(masa[corte])
).plot()
```
grafica los pares de muones seleccionados.
````
`````

`````{tip}
# Ejercicio 3 (10 minutos)

Grafica la única masa candidata por evento que sea estrictamente la más cercana a la masa del Z.

En lugar de simplemente tomar la masa máxima de cada evento, encuentra la que tenga la diferencia mínima entre la masa calculada y `masa_z = 91`.

Sugerencia: usa [`ak.argmin`](https://awkward-array.org/doc/main/reference/generated/ak.argmin.html) con `keepdims=True`. También podrías necesitar [`ak.ravel`](https://awkward-array.org/doc/main/reference/generated/ak.ravel.html)/[`ak.flatten`](https://awkward-array.org/doc/main/reference/generated/ak.flatten.html) y [`ak.drop_none`](https://awkward-array.org/doc/main/reference/generated/ak.drop_none.html).

Anticipando una de las lecciones futuras, podrías obtener una masa más precisa preguntándole a la librería Particle:

```python
import particle, hepunits

masa_z = particle.Particle.findall("Z0")[0].mass / hepunits.GeV
```

````{note}
:class: dropdown
# Solución (¡no hagas trampa!)

En lugar de maximizar `masa`, queremos minimizar `abs(masa - masa_z)` y aplicar esa elección a `masa`. [ak.argmin](https://awkward-array.org/doc/main/reference/generated/ak.argmin.html) devuelve la *posición del índice* de esta diferencia mínima, que luego podemos aplicar a la `masa` original. Sin embargo, sin `keepdims=True`, [ak.argmin](https://awkward-array.org/doc/main/reference/generated/ak.argmin.html) elimina la dimensión que necesitaríamos para que este array tenga la misma forma anidada que `masa`. Por lo tanto, usamos `keepdims=True` y luego [`ak.ravel`](https://awkward-array.org/doc/main/reference/generated/ak.ravel.html) para aplanar la lista y [`ak.drop_none`](https://awkward-array.org/doc/main/reference/generated/ak.drop_none.html) para eliminar los valores faltantes.

```python
cual = ak.argmin(abs(masa - masa_z), axis=1, keepdims=True)

hist.Hist(hist.axis.Regular(120, 0, 120, label="masa [GeV]")).fill(
    ak.drop_none(ak.ravel(masa[cual]))
).plot();
```
````
`````